[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/polars-certified/notebooks/day-04-groupby-aggregations-and-window-function.ipynb#scrollTo=20a1b2c3)

---
# Day 4 · GroupBy, Aggregations and Window Functions
**certified-journeys / polars-certified** &nbsp;|&nbsp; Analytics

> **Goal for today:** Master Polars `group_by`, multi-expression aggregations, and window functions for grouped analytics.

---
## GroupBy vs Window functions — when to use each

| Operation | Method | Output rows | Use when |
|---|---|---|---|
| Aggregate groups | `group_by().agg()` | One row per group | Summaries, reports |
| Group metric on every row | `pl.col().over()` | Same number of rows as input | Grouped transforms, ranks |
| Rolling calculation | `pl.col().rolling()` | Same number of rows as input | Time-series, moving averages |
| Split into sub-DataFrames | `partition_by()` | List of DataFrames | Per-group file export, modelling |

> **Tip:** Polars window functions (`over()`) are significantly faster than Pandas `groupby().transform()`. Use them for grouped calculations that must keep the original row count.

In [ ]:
%pip install -q polars numpy

---
## Step 1 · Build the synthetic sales dataset

We'll use a 500-row sales dataset throughout today. It has the columns that appear in real analytics work:
product, region, date, revenue, and quantity.

In [ ]:
import polars as pl
import random
from datetime import date, timedelta

random.seed(42)
products = ["laptop", "monitor", "keyboard", "mouse", "headset"]
regions  = ["north", "south", "east", "west"]

start_date = date(2024, 1, 1)

# Build list of dicts, then construct DataFrame
rows = [
    {
        "sale_id":  i,
        "product":  random.choice(products),
        "region":   random.choice(regions),
        "date":     start_date + timedelta(days=random.randint(0, 364)),
        "revenue":  round(random.uniform(20.0, 2000.0), 2),
        "quantity": random.randint(1, 10),
    }
    for i in range(1, 501)
]

df = pl.DataFrame(rows).sort("date")   # sort by date for time-series ops later
print(f"Shape: {df.shape}")
print(df.head(5))
print("\nSchema:", df.schema)

**What just happened?**
- `date` was inferred as the Python `datetime.date` type, stored as Polars `Date` dtype
- Sorting by date early makes rolling window operations (Step 4) produce correct results
- **All 500 rows were constructed as a list of dicts then passed to `pl.DataFrame()`** — clean pattern for synthetic data
- Notice `revenue` is `Float64` and `quantity` is `Int64` — Polars inferred these correctly
- The DataFrame is immutable — `sort("date")` returned a new sorted DataFrame, `df` now holds that

---
## Step 2 · `group_by().agg()` — all aggregations in one call

Polars lets you compute any number of aggregations in a single `.agg()` call.
This is more concise than Pandas' `.agg({"col": ["sum", "mean"]})` and faster —
all aggregations run in one pass over the data.

In [ ]:
# Single-column group_by with all standard aggregations
summary = (
    df
    .group_by("product")
    .agg(
        pl.col("revenue").count().alias("n_sales"),      # row count per group
        pl.col("revenue").sum().alias("total_revenue"),   # sum
        pl.col("revenue").mean().alias("avg_revenue"),    # mean
        pl.col("revenue").min().alias("min_revenue"),     # min
        pl.col("revenue").max().alias("max_revenue"),     # max
        pl.col("revenue").median().alias("med_revenue"),  # median
        pl.col("revenue").std().alias("std_revenue"),     # standard deviation
        pl.col("quantity").sum().alias("total_units"),    # different column, same group
    )
    .sort("product")
)
print("All aggregations in one agg() call:")
print(summary)

print()
# Multi-column group_by
by_product_region = (
    df
    .group_by(["product", "region"])   # two grouping keys
    .agg(
        pl.col("revenue").sum().alias("total_revenue"),
        pl.col("quantity").sum().alias("total_units"),
    )
    .sort(["product", "region"])
)
print("Multi-column group_by (first 8 rows):")
print(by_product_region.head(8))

**What just happened?**
- **All 8 aggregations ran in a single pass** over the data — no repeated iteration per aggregation
- `.alias()` names the output column inline — no separate `.rename()` needed after the fact
- Multi-column `group_by` just takes a list of column names — straightforward
- `group_by()` output is **not ordered** by default — always `.sort()` if you need stable output
- You can mix aggregations from different columns in the same `.agg()` call — they all share the group keys

---
## Step 3 · Window functions with `over()` — grouped analytics without aggregation

`over()` is Polars' equivalent of Pandas `groupby().transform()`. It computes a grouped metric
but **broadcasts the result back to every row** — the output DataFrame keeps the original row count.

Use `over()` whenever you need a grouped metric *alongside* the original data.

In [ ]:
# over() — broadcast a group-level metric to every row in that group
df_enriched = df.with_columns(
    # Total revenue per product — same value on every row of that product
    pl.col("revenue").sum().over("product").alias("product_total_revenue"),

    # Average revenue per region
    pl.col("revenue").mean().over("region").alias("region_avg_revenue"),

    # Rank within product group (1 = highest revenue sale for that product)
    pl.col("revenue").rank(descending=True).over("product").alias("rank_in_product"),

    # % contribution of this sale to its product's total
    (pl.col("revenue") / pl.col("revenue").sum().over("product") * 100)
      .round(2)
      .alias("pct_of_product"),
)

print("window functions with over() — all rows kept:")
print(df_enriched.select(
    pl.col("sale_id", "product", "region", "revenue",
           "product_total_revenue", "region_avg_revenue",
           "rank_in_product", "pct_of_product")
).head(10))

print(f"\nOriginal rows: {len(df)} | Enriched rows: {len(df_enriched)}  (same!)")

**What just happened?**
- `over("product")` computed `sum()` per product group and **broadcast it back** to every row of that product
- The output has **the same number of rows** as the input — this is the key difference from `group_by().agg()`
- **`rank().over()`** computes within-group ranks — replacing a complex Pandas `groupby().rank()` chain
- The `% contribution` expression divides each row's revenue by its group's total — a common analytics pattern
- All four `over()` expressions run in a **single `with_columns()` call** — one pass, fully parallel

---
## Step 4 · Rolling windows and EWM

Rolling functions compute metrics over a sliding window of preceding rows. In Polars they work
on sorted data and can be combined with `over()` for group-aware rolling calculations.

**The data must be sorted** before applying rolling functions — we sorted by date in Step 1.

In [ ]:
# Daily aggregation first — makes rolling more meaningful
daily = (
    df
    .group_by("date")
    .agg(pl.col("revenue").sum().alias("daily_revenue"))
    .sort("date")                      # must be sorted for rolling to be correct
)
print("Daily revenue (first 10 days):")
print(daily.head(10))

print()
# 7-day rolling mean
daily_rolling = daily.with_columns(
    pl.col("daily_revenue")
      .rolling_mean(window_size=7)          # rolling mean over 7 preceding rows
      .alias("rolling_7d_mean"),

    pl.col("daily_revenue")
      .rolling_sum(window_size=7)           # rolling sum over 7 preceding rows
      .alias("rolling_7d_sum"),

    pl.col("daily_revenue")
      .ewm_mean(span=7)                     # exponentially weighted mean (more weight to recent)
      .alias("ewm_7d"),
)
print("Rolling window results (first 12 days):")
print(daily_rolling.head(12))

**What just happened?**
- `rolling_mean(window_size=7)` computes a simple moving average — first 6 rows are `null` because there aren't 7 preceding rows yet
- `ewm_mean(span=7)` gives more weight to recent values — no warm-up nulls, starts from row 1
- **Rolling functions require sorted data** — always sort by time column before applying them
- The result has the **same number of rows** as the input — rolling is a row-wise transform, not an aggregation
- `rolling_sum`, `rolling_min`, `rolling_max`, `rolling_std` are all available with the same API

---
## Step 5 · 7-day rolling average by product group

Combine `rolling` with `over()` to compute rolling metrics **within each group** — a very common pattern
in time-series analytics that would require complex Pandas code but is concise in Polars.

In [ ]:
# Daily revenue per product — one row per product per date
product_daily = (
    df
    .group_by(["product", "date"])
    .agg(pl.col("revenue").sum().alias("daily_revenue"))
    .sort(["product", "date"])             # sort by product then date
)
print(f"Product-daily shape: {product_daily.shape}")
print(product_daily.head(8))

print()
# 7-day rolling mean per product, using sort_keys in over()
product_rolling = product_daily.with_columns(
    pl.col("daily_revenue")
      .rolling_mean(window_size=7)          # rolling within the window
      .over("product")                      # computed independently for each product group
      .alias("rolling_7d_by_product"),
)

# Show results for one product to verify correctness
laptop_rolling = product_rolling.filter(pl.col("product") == "laptop").head(15)
print("Laptop — 7-day rolling mean:")
print(laptop_rolling)

**What just happened?**
- `rolling_mean(7).over("product")` computes a **per-product** rolling mean — laptop's window doesn't mix with monitor's rows
- This is equivalent to a complex Pandas `df.groupby("product").apply(lambda g: g["daily_revenue"].rolling(7).mean())` — but a single expression
- The first 6 rows for each product are `null` — the window doesn't have 7 rows yet
- **Sorting before `over()` with rolling is critical** — the rows within each group must be in date order
- This pattern scales to any number of groups and any window size without code changes

---
## Step 6 · `partition_by()` — split a DataFrame into a list

`partition_by()` splits a DataFrame into a list of DataFrames, one per unique combination
of the partition keys. Useful for writing per-group files, running per-group models, or any
operation that genuinely needs separate DataFrames.

In [ ]:
# Partition by product — returns a list of DataFrames
parts = df.partition_by("product")          # returns List[DataFrame]

print(f"Number of partitions: {len(parts)}")  # one per unique product
for part in sorted(parts, key=lambda p: p["product"][0]):
    product_name = part["product"][0]        # get product name from first row
    print(f"  {product_name}: {len(part)} rows")

print()
# Partition by two columns
parts_2 = df.partition_by(["product", "region"], include_key=True)
print(f"Partitions by product+region: {len(parts_2)}")

# as_dict=True returns a dict keyed by the partition values — easier to look up
parts_dict = df.partition_by("product", as_dict=True)
print("\nLaptop partition:")
print(parts_dict["laptop"].head(3))

**What just happened?**
- `partition_by()` returns a **Python list of DataFrames** — one per group value (or combination)
- `as_dict=True` returns a dict keyed by partition value — convenient for direct lookup by group key
- **Use `partition_by()` when you genuinely need separate DataFrames** — e.g. writing each product to its own Parquet file
- For in-place grouped calculations, prefer `over()` — it's more memory-efficient than partitioning
- `include_key=True` (default) keeps the partition key columns in each sub-DataFrame

---
## Challenge — do this yourself

Using the `df` sales DataFrame, calculate the **% contribution of each product to its region's total revenue** using **window functions only** (no `group_by().agg()` + join).

Expected output columns: `product`, `region`, `revenue`, `region_total`, `pct_of_region`

In [ ]:
# Your solution here — use with_columns + over()
result = (
    df
    .with_columns(
        # TODO: compute region_total = sum of revenue over "region" partition
        # TODO: compute pct_of_region = revenue / region_total * 100, round to 2
    )
    .select(pl.col("product", "region", "revenue", "region_total", "pct_of_region"))
    .sort(["region", "pct_of_region"], descending=[False, True])
)
print(result.head(10))

# Verify: pct_of_region should sum to 100 for each region
# verification = result.group_by("region").agg(pl.col("pct_of_region").sum())
# print(verification.sort("region"))

---
## Day 4 key concepts recap

| Concept | What to remember |
|---|---|
| `group_by().agg()` | All aggregations in one pass — pass a list of expressions to `.agg()` |
| `.alias()` in agg | Name output columns inline — no separate rename step |
| `over()` | Window function — computes group metric and broadcasts to every row |
| `rolling_mean()` / `rolling_sum()` | Sliding window — data must be sorted first |
| `ewm_mean()` | Exponentially weighted mean — more weight to recent values |
| `rolling_mean().over()` | Per-group rolling window — most powerful pattern for time-series by group |
| `partition_by()` | Splits DataFrame into a list — use for per-group file I/O or modelling |

> **Tip:** Polars window functions (`over()`) are significantly faster than Pandas `groupby().transform()`. Use them for grouped calculations that must preserve the original row count.

---
## What's next

**Day 5** → String operations, dates, and data types: `.str.*`, `.dt.*`, Categorical dtype, schema inspection and type coercion.

Mark Day 4 complete in your [tracker](../index.html).